In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

PRECIP_PATH = Path(
    "../data/raw/precipitation/CanHomPv2_Dly_Pub/AdjTo_6158355_dly.txt"
)

colspecs = [
    (0, 5),    # year
    (5, 11),   # month
    (11, 15),  # day
    (15, 23),  # HomP
    (23, 27),  # HomP flag
    (27, 37),  # GFQCdP
    (37, 41),  # GFQCdP flag
    (41, 51),  # AdjP
    (51, 64)   # SourceClimID
]

columns = [
    "year",
    "month",
    "day",
    "HomP",
    "HomP_flag",
    "GFQCdP",
    "GFQCdP_flag",
    "AdjP",
    "SourceClimID"
]

precip_df = pd.read_fwf(
    PRECIP_PATH,
    skiprows=3,
    colspecs=colspecs,
    names=columns
)

display(precip_df.head())
print(precip_df.shape)
print(precip_df.dtypes)

,year,month,day,HomP,HomP_flag,GFQCdP,GFQCdP_flag,AdjP,SourceClimID
0,1900,1,2,64,NaN,64,NaN,64,6158350
1,1900,1,3,6,NaN,6,NaN,6,6158350
2,1900,1,4,1,NaN,1,NaN,1,6158350
3,1900,1,5,0,NaN,0,NaN,0,6158350
4,1900,1,6,0,NaN,0,NaN,0,6158350


(45471, 9)
year            int64
month           int64
day             int64
HomP            int64
HomP_flag         str
GFQCdP          int64
GFQCdP_flag       str
AdjP            int64
SourceClimID    int64
dtype: object


In [3]:
precip_df["HomP"] = precip_df["HomP"].replace(-9999, np.nan)

# 0.1 mm → mm
precip_df["precip_mm"] = precip_df["HomP"] / 10

In [4]:
precip_df["date"] = pd.to_datetime(
    precip_df[["year", "month", "day"]]
)

In [5]:
print("Date range:",
      precip_df["date"].min(),
      "→",
      precip_df["date"].max())

print("Missing precipitation:",
      precip_df["precip_mm"].isna().sum())

display(
    precip_df[
        ["date", "precip_mm", "SourceClimID"]
    ].head()
)

Date range: 1900-01-02 00:00:00 → 2024-06-30 00:00:00
Missing precipitation: 0


,date,precip_mm,SourceClimID
0,1900-01-02,6.4,6158350
1,1900-01-03,0.6,6158350
2,1900-01-04,0.1,6158350
3,1900-01-05,0.0,6158350
4,1900-01-06,0.0,6158350


In [6]:
precip_df["precip_mm"] = precip_df["HomP"] / 10

In [7]:
print(precip_df["HomP"].min())
print(precip_df["HomP"].max())
print(precip_df["HomP"].isna().sum())

0
975
0


In [8]:
precip_analysis = precip_df[
    precip_df["date"].between("1954-01-01", "2023-12-31")
].copy()

In [10]:
import calendar

precip_quality = (
    precip_analysis.groupby("year")
    .agg(
        observed_days=("date", "count"),
        valid_days=("precip_mm", "count")
    )
    .reset_index()
)

precip_quality["expected_days"] = precip_quality["year"].apply(
    lambda y: 366 if calendar.isleap(y) else 365
)

precip_quality["coverage_pct"] = (
    precip_quality["valid_days"]
    / precip_quality["expected_days"]
    * 100
)

precip_quality["usable"] = precip_quality["coverage_pct"] >= 90

display(precip_quality.head())
display(precip_quality.tail())

,year,observed_days,valid_days,expected_days,coverage_pct,usable
0,1954,365,365,365,100.0,True
1,1955,365,365,365,100.0,True
2,1956,366,366,366,100.0,True
3,1957,365,365,365,100.0,True
4,1958,365,365,365,100.0,True


,year,observed_days,valid_days,expected_days,coverage_pct,usable
65,2019,365,365,365,100.0,True
66,2020,366,366,366,100.0,True
67,2021,365,365,365,100.0,True
68,2022,365,365,365,100.0,True
69,2023,365,365,365,100.0,True


In [11]:
annual_precip = (
    precip_analysis.groupby("year")
    .agg(
        annual_precip_mm=("precip_mm", "sum")
    )
    .reset_index()
)

annual_precip = annual_precip.merge(
    precip_quality[["year", "usable"]],
    on="year",
    how="left"
)

annual_precip.loc[
    ~annual_precip["usable"],
    "annual_precip_mm"
] = np.nan

display(annual_precip.head())

,year,annual_precip_mm,usable
0,1954,941.8,True
1,1955,910.9,True
2,1956,957.1,True
3,1957,937.7,True
4,1958,740.4,True


In [12]:
from scipy.stats import linregress

precip_trend_data = annual_precip.dropna(
    subset=["annual_precip_mm"]
)

precip_result = linregress(
    precip_trend_data["year"],
    precip_trend_data["annual_precip_mm"]
)

print(
    "Precipitation trend per decade:",
    precip_result.slope * 10,
    "mm/decade"
)

print("p-value:", precip_result.pvalue)
print("R²:", precip_result.rvalue ** 2)

Precipitation trend per decade: -8.919202169538972 mm/decade
p-value: 0.20916563629748838
R²: 0.023093984509691568


In [13]:
print(
    "Average annual precipitation:",
    annual_precip["annual_precip_mm"].mean(),
    "mm"
)

print("\nWettest years:")
display(
    annual_precip.nlargest(5, "annual_precip_mm")[
        ["year", "annual_precip_mm"]
    ]
)

print("\nDriest years:")
display(
    annual_precip.nsmallest(5, "annual_precip_mm")[
        ["year", "annual_precip_mm"]
    ]
)

Average annual precipitation: 886.8757142857143 mm

Wettest years:


,year,annual_precip_mm
23,1977,1245.7
42,1996,1164.5
59,2013,1151.4
32,1986,1133.3
28,1982,1044.0



Driest years:


,year,annual_precip_mm
53,2007,634.0
50,2004,646.8
62,2016,685.7
9,1963,686.2
34,1988,724.7


In [14]:
from scipy.stats import linregress

precip_trend_data = annual_precip.dropna(
    subset=["annual_precip_mm"]
)

precip_result = linregress(
    precip_trend_data["year"],
    precip_trend_data["annual_precip_mm"]
)

print(
    f"Trend: {precip_result.slope * 10:.2f} mm/decade"
)

print(
    f"p-value: {precip_result.pvalue:.4f}"
)

print(
    f"R²: {precip_result.rvalue**2:.3f}"
)

Trend: -8.92 mm/decade
p-value: 0.2092
R²: 0.023
